# This Notebook downloads Gaia data and excludes sources as defined in the Methods. This leaves stars in Gaia with accurate astrometry in the LMC which could potentially be runaways.

The purpose of this file: query Gaia around SN1987A, apply the same Gaia-quality and LMC-membership cuts, and save a clean candidate table.

# Expected runtime: 5-15 minutes on an average Desktop / Laptop

In [1]:
import sys
print(sys.executable)

C:\Users\bukow\anaconda3\envs\mygaia_win_min\python.exe


In [2]:
import sys
import subprocess

print(sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "gaiadr3-zeropoint"
])

C:\Users\bukow\anaconda3\envs\mygaia_win_min\python.exe


0

In [3]:
from zero_point import zpt
zpt.load_tables()
print("works")

works


diagnostic cell to see which import is slow:

In [ ]:
import time
import importlib

modules = [
    "numpy",
    "pandas",
    "matplotlib.pyplot",
    "os",
    "astroquery.gaia",
    "astropy",
    "astroquery",
    "zero_point.zpt",
    "input_files.Gaia_DR3_filtering",
    "input_files.Gaia_DR3_filtering_nozeropoint",
]

for module in modules:
    t0 = time.perf_counter()
    try:
        importlib.import_module(module)
        print(f"{module:45s} ok   {time.perf_counter() - t0:.2f} s")
    except Exception as e:
        print(f"{module:45s} FAIL {time.perf_counter() - t0:.2f} s")
        print(type(e).__name__, e)

numpy                                         ok   0.00 s
pandas                                        ok   0.49 s
matplotlib.pyplot                             ok   0.45 s
os                                            ok   0.00 s


In [ ]:
import numpy as np
import pandas as ps
import matplotlib.pyplot as plt
import os

from input_files import Gaia_DR3_filtering as gdf
from input_files import Gaia_DR3_filtering_nozeropoint as gdf_nozp
from astroquery.gaia import Gaia

print("all imports ok")
%matplotlib inline

# File directory

In [ ]:
from pathlib import Path

output_dir = Path("output_files")
input_files_dir = Path("input_files")

output_dir.mkdir(exist_ok=True)
input_files_dir.mkdir(exist_ok=True)

ra_87a = 83.86661833333333
dec_87a = -69.26975372222222

radius_deg = 0.10   # first run: about 86 pc
g_limit = 19.0      # use 18.0 if you want to stay closer to Stoop

In [ ]:
parm_circle = [279.4650108, -31.6718983, 2.0]

### Own debugging attempts

In [ ]:
from pathlib import Path

raw_file = Path(input_files_dir) / "R136_runaways_raw.csv"

print(raw_file.exists())
if raw_file.exists():
    print(raw_file.resolve())
    print(raw_file.stat().st_size / 1e6, "MB")

# Download Gaia data
# This may take 10 minutes

Gaia download. It is like the Stoop query, but centered on SN1987A and using an ICRS circle instead of a Galactic l/b rectangle:

In [ ]:
from astroquery.gaia import Gaia

raw_file = input_files_dir / "SN1987A_raw.csv"

tot_string = f"""
SELECT
    *,
    tmass.j_m,
    tmass.j_msigcom,
    tmass.h_m,
    tmass.h_msigcom,
    tmass.ks_m,
    tmass.ks_msigcom
FROM gaiadr3.gaia_source AS dr3
LEFT JOIN gaiadr3.tmass_psc_xsc_best_neighbour AS xmatch
    USING (source_id)
LEFT JOIN gaiadr3.tmass_psc_xsc_join AS xjoin
    USING (clean_tmass_psc_xsc_oid)
LEFT JOIN gaiadr1.tmass_original_valid AS tmass
    ON xjoin.original_psc_source_id = tmass.designation
WHERE 1 = CONTAINS(
    POINT('ICRS', dr3.ra, dr3.dec),
    CIRCLE('ICRS', {ra_87a}, {dec_87a}, {radius_deg})
)
AND dr3.phot_g_mean_mag < {g_limit}
"""

if raw_file.exists():
    print("Raw file already exists:", raw_file)
else:
    job = Gaia.launch_job_async(
        tot_string,
        dump_to_file=True,
        output_format="csv",
        output_file=str(raw_file)
    )

In [ ]:
runaways_raw = ps.read_csv(input_files_dir / "SN1987A_raw.csv", low_memory=False)
runaways_all = runaways_raw.copy()

### Own debugging code part 2

In [ ]:
print(runaways_raw.shape)
runaways_raw.head()

Alternative for the runaways_raw runaways_all cell: This tells pandas to inspect the file more consistently before deciding column types. It will use more memory, but with a 644 MB file that is expected.

In [ ]:
runaways_raw = ps.read_csv(
    input_files_dir + 'R136_runaways_raw.csv',
    low_memory=False
)

runaways_all = runaways_raw.copy()

To find which column caused it:

In [ ]:
cols = ps.read_csv(input_files_dir + 'R136_runaways_raw.csv', nrows=1).columns
print(cols[158])

# Galactic coordinates

In [ ]:
def icrs_to_gal(ra, ra_error, dec, dec_error, parallax, parallax_error, pmra, pmra_error, pmdec, pmdec_error,
                ra_dec_corr, ra_parallax_corr, ra_pmra_corr, ra_pmdec_corr, dec_parallax_corr, dec_pmra_corr,
                dec_pmdec_corr, parallax_pmra_corr, parallax_pmdec_corr, pmra_pmdec_corr):
    
    A_prime_G = np.array([[-0.0548755604162154, -0.8734370902348850, -0.4838350155487132], [0.4941094278755837, -0.4448296299600112, 0.7469822444972189], [-0.8676661490190047, -0.1980763734312015, 0.4559837761750669]])
    
    # icrs position vector
    r_icrs = np.array([np.cos(np.radians(ra))*np.cos(np.radians(dec)), np.sin(np.radians(ra))*np.cos(np.radians(dec)), np.sin(np.radians(dec))])
    
    # galactic position vector
    r_gal_0 = np.sum(A_prime_G[0,:]*r_icrs)
    r_gal_1 = np.sum(A_prime_G[1,:]*r_icrs)
    r_gal_2 = np.sum(A_prime_G[2,:]*r_icrs)
                      
    # calculate galactic l and b
    l = np.degrees(np.arctan2(r_gal_1, r_gal_0))
    b = np.degrees(np.arctan2(r_gal_2, np.sqrt(r_gal_0**2 + r_gal_1**2)))
    
    p_icrs = np.array([-np.sin(np.radians(ra)), np.cos(np.radians(ra)), 0.])
    q_icrs = np.array([-np.cos(np.radians(ra))*np.sin(np.radians(dec)), -np.sin(np.radians(ra))*np.sin(np.radians(dec)), np.cos(np.radians(dec))])
    
    p_gal = np.array([-np.sin(np.radians(l)), np.cos(np.radians(l)), 0.])
    q_gal = np.array([-np.cos(np.radians(l))*np.sin(np.radians(b)), -np.sin(np.radians(l))*np.sin(np.radians(b)), np.cos(np.radians(b))])
    
    pm_icrs = p_icrs*pmra + q_icrs*pmdec
    
    pm_gal_0 = np.sum(A_prime_G[0,:]*pm_icrs)
    pm_gal_1 = np.sum(A_prime_G[1,:]*pm_icrs)
    pm_gal_2 = np.sum(A_prime_G[2,:]*pm_icrs)
    
    # calculate galactic pml and pmb
    pml = p_gal[0]*pm_gal_0 + p_gal[1]*pm_gal_1 + p_gal[2]*pm_gal_2
    pmb = q_gal[0]*pm_gal_0 + q_gal[1]*pm_gal_1 + q_gal[2]*pm_gal_2
    
    matr_icrs = np.array([[p_icrs[0], q_icrs[0]], [p_icrs[1], q_icrs[1]], [p_icrs[2], q_icrs[2]]])
    matr_gal_prime = np.array([[p_gal[0], q_gal[0]], [p_gal[1], q_gal[1]], [p_gal[2], q_gal[2]]]).T
    
    # obtain G, describing ICRS to galactic rotation
    G = np.matmul(matr_gal_prime, np.matmul(A_prime_G, matr_icrs))
    
    # obtain J, the Jacobian of the transformation
    J = np.array([[G[0,0], G[0,1], 0, 0, 0],
                 [G[1,0], G[1,1], 0, 0, 0],
                 [0, 0, 1, 0, 0], 
                 [0, 0, 0, G[0,0], G[0,1]],
                 [0, 0, 0, G[1,0], G[1,1]]])
    
    # ICRS covariance matrix
    C = np.array([[ra_error**2, ra_error*dec_error*ra_dec_corr, ra_error*parallax_error*ra_parallax_corr, ra_error*pmra_error*ra_pmra_corr, ra_error*pmdec_error*ra_pmdec_corr],
                 [dec_error*ra_error*ra_dec_corr, dec_error**2, dec_error*parallax_error*dec_parallax_corr, dec_error*pmra_error*dec_pmra_corr, dec_error*pmdec_error*dec_pmdec_corr],
                 [parallax_error*ra_error*ra_parallax_corr, parallax_error*dec_error*dec_parallax_corr, parallax_error**2, parallax_error*pmra_error*parallax_pmra_corr, parallax_error*pmdec_error*parallax_pmdec_corr],
                 [pmra_error*ra_error*ra_pmra_corr, pmra_error*dec_error*dec_pmra_corr, pmra_error*parallax_error*parallax_pmra_corr, pmra_error**2, pmra_error*pmdec_error*pmra_pmdec_corr],
                 [pmdec_error*ra_error*ra_pmdec_corr, pmdec_error*dec_error*dec_pmdec_corr, pmdec_error*parallax_error*parallax_pmdec_corr, pmdec_error*pmra_error*pmra_pmdec_corr, pmdec_error**2]])
    
    
    C_gal = np.matmul(J, np.matmul(C, J.T))
    
    l_error = np.sqrt(C_gal[0,0])
    b_error = np.sqrt(C_gal[1,1])
    pml_error = np.sqrt(C_gal[3,3])
    pmb_error = np.sqrt(C_gal[4,4])
    l_b_corr = C_gal[0,1]/(l_error*b_error)
    l_parallax_corr = C_gal[0,2]/(l_error*parallax_error)
    l_pml_corr = C_gal[0,3]/(l_error*pml_error)
    l_pmb_corr = C_gal[0,4]/(l_error*pmb_error)
    b_parallax_corr = C_gal[1,2]/(b_error*parallax_error)
    b_pml_corr = C_gal[1,3]/(b_error*pml_error)
    b_pmb_corr = C_gal[1,3]/(b_error*pmb_error)
    parallax_pml_corr = C_gal[2,3]/(parallax_error*pml_error)
    parallax_pmb_corr = C_gal[2,4]/(parallax_error*pmb_error)
    pml_pmb_corr = C_gal[3,4]/(pml_error*pmb_error)
    
    return l, l_error, b, b_error, pml, pml_error, pmb, pmb_error, l_b_corr, l_parallax_corr, l_pml_corr, l_pmb_corr, b_parallax_corr, b_pml_corr, b_pmb_corr, parallax_pml_corr, parallax_pmb_corr, pml_pmb_corr


# Transform Equatorial to Galactic coordinates

In [ ]:
runaways_all['n_iter'] = np.arange(len(runaways_all))
print(runaways_all)

In [ ]:
l_column = np.zeros(len(runaways_all))
l_error_column = np.zeros(len(runaways_all))
b_column = np.zeros(len(runaways_all))
b_error_column = np.zeros(len(runaways_all))
pml_column = np.zeros(len(runaways_all))
pml_error_column = np.zeros(len(runaways_all))
pmb_column = np.zeros(len(runaways_all))
pmb_error_column = np.zeros(len(runaways_all))
l_b_corr_column = np.zeros(len(runaways_all))
l_parallax_corr_column = np.zeros(len(runaways_all))
l_pml_corr_column = np.zeros(len(runaways_all))
l_pmb_corr_column = np.zeros(len(runaways_all))
b_parallax_corr_column = np.zeros(len(runaways_all))
b_pml_corr_column = np.zeros(len(runaways_all))
b_pmb_corr_column = np.zeros(len(runaways_all))
parallax_pml_corr_column = np.zeros(len(runaways_all))
parallax_pmb_corr_column = np.zeros(len(runaways_all))
pml_pmb_corr_column = np.zeros(len(runaways_all))

ii = 0

runaways_all = runaways_all.sort_values('ra')
for index, row in runaways_all.sort_values('ra').iterrows():
    l, l_error, b, b_error, pml, pml_error, pmb, pmb_error, l_b_corr, l_parallax_corr, l_pml_corr, l_pmb_corr, b_parallax_corr, b_pml_corr, b_pmb_corr, parallax_pml_corr, parallax_pmb_corr, pml_pmb_corr = icrs_to_gal(row.ra, row.ra_error, row.dec, row.dec_error, row.parallax, row.parallax_error, row.pmra, row.pmra_error, row.pmdec, row.pmdec_error, row.ra_dec_corr, row.ra_parallax_corr, row.ra_pmra_corr, row.ra_pmdec_corr, row.dec_parallax_corr, row.dec_pmra_corr, row.dec_pmdec_corr, row.parallax_pmra_corr, row.parallax_pmdec_corr, row.pmra_pmdec_corr)
    l_column[ii] = l
    l_error_column[ii] = l_error
    b_column[ii] = b
    b_error_column[ii] = b_error
    pml_column[ii] = pml
    pml_error_column[ii] = pml_error
    pmb_column[ii] = pmb
    pmb_error_column[ii] = pmb_error
    l_b_corr_column[ii] = l_b_corr
    l_parallax_corr_column[ii] = l_parallax_corr
    l_pml_corr_column[ii] = l_pml_corr
    l_pmb_corr_column[ii] = l_pmb_corr
    b_parallax_corr_column[ii] = b_parallax_corr
    b_pml_corr_column[ii] = b_pml_corr
    b_pmb_corr_column[ii] = b_pmb_corr
    parallax_pml_corr_column[ii] = parallax_pml_corr
    parallax_pmb_corr_column[ii] = parallax_pmb_corr
    pml_pmb_corr_column[ii] = pml_pmb_corr
    if ii%1000 == 0:
        print(ii)
    ii+=1

runaways_all['l'] = l_column
runaways_all['l'] = runaways_all.l + 360
runaways_all['l_error'] = l_error_column
runaways_all['b'] = b_column
runaways_all['b_error'] = b_error_column
runaways_all['pml'] = pml_column
runaways_all['pml_error'] = pml_error_column
runaways_all['pmb'] = pmb_column
runaways_all['pmb_error'] = pmb_error_column
runaways_all['l_b_corr'] = l_b_corr_column
runaways_all['l_parallax_corr'] = l_parallax_corr_column
runaways_all['l_pml_corr'] = l_pml_corr_column
runaways_all['l_pmb_corr'] = l_pmb_corr_column
runaways_all['b_parallax_corr'] = b_parallax_corr_column
runaways_all['b_pml_corr'] = b_pml_corr_column
runaways_all['b_pmb_corr'] = b_pmb_corr_column
runaways_all['parallax_pml_corr'] = parallax_pml_corr_column
runaways_all['parallax_pmb_corr'] = parallax_pmb_corr_column
runaways_all['pml_pmb_corr'] = pml_pmb_corr_column




# Apply the Gaia zero point offset

In [ ]:
print(len(runaways_all))
runaways_all = gdf.gaia_filtering(runaways_all, 1e10, 1e10, -1e10, 1e10, 1e10)
print(len(runaways_all))

^ ot an error. It is a warning from the Gaia parallax zero-point correction package. the function completed. Nothing crashed.

The warnings mean: for some Gaia sources, the values used to compute the parallax zero-point correction are outside the calibrated range of the zero_point package: These warnings are common when applying Gaia zero-point corrections to a large raw catalogue. The Stoop et al. method does correct the Gaia parallax zero-point and then applies Gaia-quality / LMC membership filters, so this warning is happening in the expected part of the workflow.

The reason no sources were removed is because this call uses deliberately huge limits: Those thresholds are basically “do not actually filter yet.” It is probably just applying zero-point correction and checking the filter machinery.

# Runaway Gaia parameters

In [ ]:
import astropy.units as u
from astropy.coordinates import SkyCoord

r_search = 1.0  # deg; broad test region

sn87a = SkyCoord("05h35m27.9884s", "-69d16m11.1134s", frame="icrs")
coords = SkyCoord(
    ra=runaways_all["ra"].values * u.deg,
    dec=runaways_all["dec"].values * u.deg,
    frame="icrs"
)

sep = sn87a.separation(coords).deg

runaways_raw = runaways_all.copy()
runaways_raw = runaways_raw[sep < r_search].copy()

print(len(runaways_raw))

In [ ]:
import astropy.units as u
from astropy.coordinates import SkyCoord

# SN1987A position, ICRS
sn87a = SkyCoord("05h35m27.9884s", "-69d16m11.1134s", frame="icrs")

center_l = sn87a.galactic.l.deg
center_b = sn87a.galactic.b.deg

r_search = 0.10  # or 0.25 / 0.50 / 1.0, depending on the run

# Compatibility with old R136 code
parm_circle = [center_l, center_b, r_search]

print("SN1987A galactic centre:", parm_circle)

In [ ]:
fig, ax = plt.subplots(1, figsize=(14, 14))

print(
    (np.amax(runaways_raw.l) - np.amin(runaways_raw.l)) /
    (np.amax(runaways_raw.b) - np.amin(runaways_raw.b))
)

ax.plot(runaways_raw.l, runaways_raw.b, 'bo', ms=1., alpha=0.5)

# Mark SN1987A
ax.plot(center_l, center_b, 'ro', ms=5., label='SN1987A')

ax.tick_params(labelsize=12)
ax.set_xlabel('Galactic longitude (deg)', fontsize=15)
ax.set_ylabel('Galactic latitude (deg)', fontsize=15)
ax.legend()

plt.show()

In [ ]:
runaways_filter = runaways_raw.copy()
runaways = gdf_nozp.gaia_filtering(runaways_filter, max_ruwe, max_ipd_frac_multi_peak, min_visibility_periods_used, 
                              max_duplicated_source, max_ipd_gof_harmonic_amplitude)


# Filter on parallax

In [ ]:
print(len(runaways))
runaways = runaways[runaways.parallax_error < 0.05]
runaways = runaways[runaways.parallax - 3*runaways.parallax_error < 1/49.59]
print(len(runaways))

# Save the runaway candidates

In [ ]:
runaways.to_csv(output_dir / "SN1987A_cands.csv", index=False)
runaways_raw.to_csv(output_dir / "SN1987A_raw_zp.csv", index=False)

finished the first stage: generating / filtering the Gaia catalogue used as input for the runaway search; checking what files the notebook created or updated:

In [ ]:
from pathlib import Path
import pandas as pd

for folder in ["input_files", "output_files"]:
    print("\n" + folder)
    for f in sorted(Path(folder).glob("*.csv")):
        print(f"{f.name:35s} {f.stat().st_size/1e6:8.2f} MB")

inspecting the main output tables

In [ ]:
for fname in [
    "input_files/R136_runaways_raw.csv",
    "output_files/runaways_cands.csv",
    "output_files/runaways_mcmc.csv",
    "output_files/runaways_outliers_mcmc.csv",
    "output_files/runaways_raw_zp_1deg.csv",
]:
    f = Path(fname)
    if f.exists():
        df = pd.read_csv(f, nrows=5)
        print("\n", fname)
        print("columns:", list(df.columns[:15]), "...")
        print(df.head())

cleaner overview:

In [ ]:
from pathlib import Path
import pandas as pd

for fname in [
    "output_files/runaways_raw_zp_1deg.csv",
    "output_files/runaways_cands.csv",
    "output_files/runaways_mcmc.csv",
    "output_files/runaways_outliers_mcmc.csv",
]:
    f = Path(fname)
    if f.exists():
        df = pd.read_csv(f)
        print(fname)
        print("shape:", df.shape)
        print("first columns:", list(df.columns[:20]))
        print()

In [ ]:
cands_file = output_dir / "SN1987A_cands.csv"

if not cands_file.exists():
    raise FileNotFoundError(
        "Missing output_files/SN1987A_cands.csv. "
        "First run the cleaned Gaia-candidate notebook."
    )

df = pd.read_csv(cands_file, low_memory=False)
df.columns = df.columns.str.strip()

coords = SkyCoord(df["ra"].values*u.deg, df["dec"].values*u.deg, frame="icrs")
sep_deg = sn87a.separation(coords).deg
sep_pc = sn87a.separation(coords).radian * D_LMC_KPC * 1000.0

df["sep_87a_deg"] = sep_deg
df["sep_87a_pc"] = sep_pc

print("Rows:", len(df))
print("Max separation from SN1987A [deg]:", np.nanmax(df["sep_87a_deg"]))
print("Max separation from SN1987A [pc]:", np.nanmax(df["sep_87a_pc"]))
print("Median RA/Dec:", np.nanmedian(df["ra"]), np.nanmedian(df["dec"]))

# For the 0.10 deg run, max separation should be about 0.10 deg.
# Leave tolerance because later you may intentionally run 0.25 or 0.50 deg.
if np.nanmax(df["sep_87a_deg"]) > 0.55:
    raise ValueError(
        "This candidate file is too wide for the current SN1987A runs. "
        "It may be an R136/raw file or a 1-2 degree test file. "
        "Regenerate SN1987A_cands.csv from SN1987A_raw.csv."
    )

df.to_csv(output_dir / "SN1987A_cands_checked.csv", index=False)
df.head()


raw Gaia query

has proper motions

RUWE < 1.4

visibility_periods_used >= 10

not duplicated_source

IPD cuts passed

parallax consistent with LMC / not foreground

within 100 pc of SN1987A

within 50 pc of SN1987A

dv_local_resid < 30 km/s

dv_local_resid < 20 km/s

In [1]:
fig, ax = plt.subplots(figsize=(6, 6))

sc = ax.scatter(
    df["bp_rp"],
    df["phot_g_mean_mag"],
    c=df["dv_local_resid_kms"],
    vmin=0,
    vmax=80,
    s=10,
    alpha=0.6
)

cand = df[
    (df["sep_87a_pc"] < 100) &
    (df["dv_local_resid_kms"] < 30)
    ]

ax.scatter(
    cand["bp_rp"],
    cand["phot_g_mean_mag"],
    facecolors="none",
    edgecolors="black",
    s=45,
    label="co-moving candidates"
)

ax.invert_yaxis()
ax.set_xlabel("BP - RP")
ax.set_ylabel("G")
ax.set_title("Gaia CMD near SN1987A coloured by PM difference")
ax.legend()

cb = plt.colorbar(sc, ax=ax)
cb.set_label("Residual velocity difference [km/s]")

NameError: name 'plt' is not defined